# Week 3, Day 2: A WrestleMania III fact desk

**Lesson:** [the Colab notebook](https://colab.research.google.com/drive/1aMaEw8A56xs0bRM4lu8z7ou18jqyybGm) linked from [week3/day2.ipynb](../../../week3/day2.ipynb), *Hugging Face pipelines*. **About 40 minutes.**

**What you practice here**

- Making a pipeline from a task name, with the task's default model or one you choose, on a device, and then calling it.
- Calling five kinds of pipeline with the inputs and options each one expects: the labels to choose from, a question and its context, and a summary's length limits.
- Reading each answer from where its task puts it: a label, the best of your labels, an answer, a summary and a translation.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import os, pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT is None and ("google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ):
    # in Colab: fetch the checker from your fork on GitHub, and the packages Colab lacks
    import subprocess, urllib.request
    ROOT = pathlib.Path("/content/fork")
    (ROOT / "plan").mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/starlest/llm_engineering/main/plan/check.py", ROOT / "plan" / "check.py")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv", "tiktoken"], check=False)
    print("In Colab, the checks run here too. To keep your work, use File > Save a copy in GitHub, to this notebook's path in your fork.")
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week3/day2", globals())
else:  # outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here.")

## The task

The lesson tours Hugging Face's pipelines. Each one is made from a task name, such as `"sentiment-analysis"`, and then
called on text, and each task gives its answer in a shape of its own. Use five of them to build a fact desk for a
paragraph about WrestleMania III: the mood of fans' comments, the paragraph's topic, answers to questions about it, a
summary, and a Spanish version.

transformers is not in the checks' Python, so every function takes transformers' `pipeline` function as its first
parameter, makes its pipeline with it, and calls that pipeline as the lesson does. The checks hand your functions a
stand-in `pipeline` that records what was made and how it was called, and answers in each task's shape.

## What each call should give

```python
fan_moods(pipeline, ["What a main event!", "The opening match was far too short."])
# makes one sentiment-analysis pipeline on "cuda", and gives ['POSITIVE', 'NEGATIVE']

match_topic(pipeline, TEXT, ["sports entertainment", "politics", "cooking"])
# 'sports entertainment': the label with the highest score

ask(pipeline, "Where did WrestleMania III take place?", TEXT)
# 'the Pontiac Silverdome'

summarize(pipeline, TEXT)
# 'WrestleMania III took place at the Pontiac Silverdome in 1987. Hulk Hogan ...'

to_spanish(pipeline, "WrestleMania III took place in Pontiac, Michigan.")
# 'WrestleMania III tuvo lugar en Pontiac, Michigan.'
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.

### 1 · `fan_moods(pipeline, comments, model=None, device="cuda")`

Tells, for each fan comment, whether it is positive or negative, with one sentiment-analysis pipeline. Make one
pipeline for the task, with `model` and `device`; with `model=None`, Hugging Face picks the task's default model. Then
run it on the comments, either once for each comment or once on the whole list: the real pipeline takes both. One
text gives a list holding one result, a dict with a `"label"` and a `"score"`, and a list of texts gives one such dict
for each text. Return the labels, in the comments' order, such as `["POSITIVE", "NEGATIVE"]`.

<details><summary>Hint</summary>

```python
analyzer = pipeline("sentiment-analysis", model=model, device=device)
labels = [result["label"] for result in analyzer(comments)]
```

</details>

In [ ]:
def fan_moods(pipeline, comments, model=None, device="cuda"):
    # Write fan_moods here, as described above.
    raise NotImplementedError("fan_moods is not written yet")

In [ ]:
check("fan_moods")

### 2 · `match_topic(pipeline, text, labels, model=None, device="cuda")`

Finds which of your labels fits a text best, with a zero-shot classification pipeline. Make the pipeline with `model`
and `device`, and call it on `text`, with `labels` as the candidate labels it chooses from. It returns a dict with the
`"labels"`, best first, and their `"scores"` in the same order. Return the label with the highest score.

<details><summary>Hint</summary>

```python
classifier = pipeline("zero-shot-classification", model=model, device=device)
result = classifier(text, candidate_labels=labels)
```

</details>

In [ ]:
def match_topic(pipeline, text, labels, model=None, device="cuda"):
    # Write match_topic here, as described above.
    raise NotImplementedError("match_topic is not written yet")

In [ ]:
check("match_topic")

### 3 · `ask(pipeline, question, context, model=None, device="cuda")`

Answers a question from a passage of text, with a question-answering pipeline. Make the pipeline with `model` and
`device`, and call it with the question and the context, both by name, as the lesson does. It returns a dict with the
answer's `"score"`, `"start"`, `"end"` and `"answer"`: return the `"answer"`.

<details><summary>Hint</summary>

```python
question_answerer = pipeline("question-answering", model=model, device=device)
return question_answerer(question=question, context=context)["answer"]
```

</details>

In [ ]:
def ask(pipeline, question, context, model=None, device="cuda"):
    # Write ask here, as described above.
    raise NotImplementedError("ask is not written yet")

In [ ]:
check("ask")

### 4 · `summarize(pipeline, text, max_length=60, min_length=20, model=None, device="cuda")`

Summarizes a text in a few sentences, with a summarization pipeline. Make the pipeline with `model` and `device`, and
call it on `text` with `max_length` and `min_length` as the summary's length limits and with sampling turned off, as
the lesson does. It returns a list holding one dict: return that dict's `"summary_text"`.

<details><summary>Hint</summary>

```python
summarizer = pipeline("summarization", model=model, device=device)
result = summarizer(text, max_length=max_length, min_length=min_length, do_sample=False)
return result[0]["summary_text"]
```

</details>

In [ ]:
def summarize(pipeline, text, max_length=60, min_length=20, model=None, device="cuda"):
    # Write summarize here, as described above.
    raise NotImplementedError("summarize is not written yet")

In [ ]:
check("summarize")

### 5 · `to_spanish(pipeline, text, model="Helsinki-NLP/opus-mt-en-es", device="cuda")`

Translates English text into Spanish, with the model the lesson chose. Make an English-to-Spanish translation
pipeline with `model` and `device`, call it on the text, and return the translated text of its first result. The
function's `model` defaults to the lesson's `Helsinki-NLP/opus-mt-en-es`, because this task has no default model of
its own.

<details><summary>Hint</summary>

```python
translator = pipeline("translation_en_to_es", model=model, device=device)
return translator(text)[0]["translation_text"]
```

</details>

In [ ]:
def to_spanish(pipeline, text, model="Helsinki-NLP/opus-mt-en-es", device="cuda"):
    # Write to_spanish here, as described above.
    raise NotImplementedError("to_spanish is not written yet")

In [ ]:
check("to_spanish")

## Check everything

The cell below runs all 7 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 3, Day 2 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. This needs a GPU, so it runs in Google Colab: push this notebook, then [open it in Colab](https://colab.research.google.com/github/starlest/llm_engineering/blob/main/plan/exercises/week3/day2.ipynb), choose a T4 runtime, and run the cells from the top. The first cell fetches the checker from your fork, so the checks run in Colab too. Write the steps in the cell at the end.

Colab works on its own copy of the notebook. To keep the Try code you write there, use **File > Save a copy in GitHub** (to this notebook's path in your fork), then run `git pull` in your codespace before you commit there again. Your parts are already in your fork, which is what the tracker reads.

The models download a few GB the first time, which Colab does quickly, and they fit on its free T4 together.

1. Keep the `!pip install` line in the first Try cell. Write a short paragraph about WrestleMania III, and three
   comments a fan might post about it, as constants.
2. Run each of your five functions on them with the real `pipeline`, and print what each returns.
3. Run `fan_moods` again with `model="nlptown/bert-base-multilingual-uncased-sentiment"`, the lesson's better
   sentiment model, and compare its labels with the default model's.
4. Optional: try a task the exercise leaves out, such as `pipeline("ner", device="cuda")` or
   `pipeline("text-generation", device="cuda")`, and look at what it returns.

<details><summary>Stuck? One way to write it</summary>

```python
!pip install -q --upgrade transformers==4.57.6 sentencepiece
from transformers import pipeline

TEXT = ("WrestleMania III took place on 29 March 1987 at the Pontiac Silverdome in Pontiac, Michigan, before a crowd "
        "the WWF announced as 93,173. In the main event Hulk Hogan body-slammed Andre the Giant and pinned him to keep "
        "the WWF Championship. Earlier, Ricky Steamboat beat Randy Savage for the Intercontinental Championship in a "
        "match many fans still call the best of the decade.")
COMMENTS = ["What a main event!", "The opening match was far too short.", "Steamboat and Savage stole the show."]

print(fan_moods(pipeline, COMMENTS))
print(fan_moods(pipeline, COMMENTS, model="nlptown/bert-base-multilingual-uncased-sentiment"))
print(match_topic(pipeline, TEXT, ["sports entertainment", "politics", "cooking"]))
print(ask(pipeline, "Who did Hulk Hogan pin?", TEXT))
print(summarize(pipeline, TEXT))
print(to_spanish(pipeline, TEXT))

# Optional: a task the exercise leaves out
ner = pipeline("ner", device="cuda")
for entity in ner(TEXT):
    print(entity)
```

</details>

In [ ]:
!pip install -q --upgrade transformers==4.57.6 sentencepiece
# Try it for real: write the steps above here, in as many cells as you like.